In [ ]:
# ============================================================
# FINAL SCALED_EI03 — ENERGY ISLAND (ECONOMIC MULTI-VECTOR)
# Offshore wind + HVDC + Electrolysis + Battery + H2 EXPORT
# ============================================================

import pypsa
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")
base_fn = root / "resources" / "networks" / "base_s_50_elec.nc"
out_fn  = root / "results" / "networks" / "EI03_energy_island.nc"

n = pypsa.Network(base_fn)
print("Loaded base network:", base_fn.name)

# ------------------------------------------------------------
# COUNTRIES
# ------------------------------------------------------------
EI_COUNTRIES      = ["DK", "DE", "NL", "NO", "SE"]
PROFILE_COUNTRIES = ["DK", "DE", "NL", "NO"]

# ------------------------------------------------------------
# NETCDF SAFETY
# ------------------------------------------------------------
if "color" in n.carriers.columns:
    n.carriers["color"] = n.carriers["color"].astype(str)

# ------------------------------------------------------------
# DISPATCH-ONLY
# ------------------------------------------------------------
for comp in ["generators", "links", "lines", "storage_units", "stores"]:
    if hasattr(n, comp):
        if "p_nom_extendable" in getattr(n, comp).columns:
            getattr(n, comp)["p_nom_extendable"] = False

if "s_nom_extendable" in n.lines.columns:
    n.lines["s_nom_extendable"] = False

print("✓ Dispatch-only mode enforced")

# ============================================================
# ENERGY ISLAND
# ============================================================

EI_BUS = "EI03_bus"

if EI_BUS not in n.buses.index:
    n.add("Bus", EI_BUS, carrier="AC", country="EI", x=9.0, y=56.5)

print("✓ Energy Island bus added")



# ------------------------------------------------------------
# AC CONNECTION TO MAINLAND (CRITICAL)
# ------------------------------------------------------------
# Tie EI03_bus electrically to one mainland bus (AC)
ref_bus = (
    n.buses[n.buses.country.isin(EI_COUNTRIES)]
    .index[0]
)

n.add(
    "Line",
    name="EI03_AC_tie",
    bus0=EI_BUS,
    bus1=ref_bus,
    s_nom=20_000,     # MW
    x=0.01,
    r=0.001,
)

print("✓ EI03 AC tie-line added (critical)")


# ------------------------------------------------------------
# CARRIERS 
# ------------------------------------------------------------
REQUIRED_CARRIERS = [
    "offwind-ac", "electrolysis", "battery",
    "hydrogen", "DC", "load_shedding"
]

for c in REQUIRED_CARRIERS:
    if c not in n.carriers.index:
        n.carriers.loc[c] = {col: 0 for col in n.carriers.columns}

print("✓ Required carriers ensured")

# ------------------------------------------------------------
# OFFSHORE WIND PROFILE (NORTH SEA)
# ------------------------------------------------------------
offwind_mask = (
    n.generators.carrier.str.contains("offwind")
    & n.generators.bus.map(n.buses.country).isin(PROFILE_COUNTRIES)
)

p_max_pu_ns = (
    n.generators_t.p_max_pu.loc[:, offwind_mask]
    .mean(axis=1)
    .clip(0.0, 1.0)
)

# ------------------------------------------------------------
# OFFSHORE WIND — 15 GW
# ------------------------------------------------------------
n.add(
    "Generator",
    "EI03_offshore_wind",
    bus=EI_BUS,
    carrier="offwind-ac",
    p_nom=15_000,
    marginal_cost=0.0,
)

n.generators_t.p_max_pu["EI03_offshore_wind"] = p_max_pu_ns.values
print("✓ Offshore wind added")

# ------------------------------------------------------------
# HVDC LINKS — 10 GW
# ------------------------------------------------------------
HVDC_CAP = 2_000

for c in EI_COUNTRIES:
    loads_c = n.loads[n.loads.bus.map(n.buses.country) == c]
    main_bus = loads_c.groupby("bus").p_set.sum().idxmax()

    n.add(
        "Link",
        f"EI03_HVDC_{c}",
        bus0=EI_BUS,
        bus1=main_bus,
        carrier="DC",
        p_nom=HVDC_CAP,
        efficiency=0.98,
        marginal_cost=1.0,
    )

print("✓ HVDC links added")

# ------------------------------------------------------------
# HYDROGEN BUS
# ------------------------------------------------------------
if "H2" not in n.buses.index:
    n.add("Bus", "H2", carrier="hydrogen", country="EI")

print("✓ Hydrogen bus added")

# ------------------------------------------------------------
# ELECTROLYSER — 5 GW
# ------------------------------------------------------------
n.add(
    "Link",
    "EI03_electrolyser",
    bus0=EI_BUS,
    bus1="H2",
    carrier="electrolysis",
    p_nom=5_000,
    efficiency=0.70,
    marginal_cost=70.0,
)

print("✓ Electrolyser added")

# ------------------------------------------------------------
# H2 STORAGE — 300 GWh
# ------------------------------------------------------------
n.add(
    "Store",
    "EI03_H2_storage",
    bus="H2",
    carrier="hydrogen",
    e_nom=300_000,   # MWh
    e_cyclic=False,
    marginal_cost=0.1,
)

print("✓ H2 storage added")

# ------------------------------------------------------------
# H2 ACTIVATION — SURPLUS-BASED (ROBUST)
# ------------------------------------------------------------

# Activate H2 only during high-wind periods (top 20%)
WIND_THRESHOLD = p_max_pu_ns.quantile(0.80)

h2_active = p_max_pu_ns.values > WIND_THRESHOLD


# ------------------------------------------------------------
# H2 EXPORT — ECONOMIC DEMAND (CONDITIONAL)
# ------------------------------------------------------------
H2_PRICE = 90.0  # €/MWh_H2

n.add(
    "Load",
    "EI03_H2_export",
    bus="H2",
    carrier="hydrogen",
    p_set=10_000 * p_max_pu_ns.values * h2_active,
    marginal_cost=-H2_PRICE,
)



print("✓ H2 export load added (THIS ENABLES ELECTROLYSIS)")


# ------------------------------------------------------------
# BATTERY — 2 GWh
# ------------------------------------------------------------
n.add(
    "StorageUnit",
    "EI03_battery",
    bus=EI_BUS,
    carrier="battery",
    p_nom=1_000,
    max_hours=2.0,
    efficiency_store=0.95,
    efficiency_dispatch=0.95,
    cyclic_state_of_charge=False,
)

print("✓ Battery added")

# ------------------------------------------------------------
# LOAD SHEDDING
# ------------------------------------------------------------
for b in n.buses.index:
    g = f"load_shed_{b}"
    if g not in n.generators.index:
        n.add(
            "Generator",
            g,
            bus=b,
            carrier="load_shedding",
            p_nom=1e6,
            marginal_cost=10_000.0,
        )

print("✓ Load shedding ensured")

# ------------------------------------------------------------
# FINAL NETCDF SAFETY
# ------------------------------------------------------------
for col in ["color", "nice_name"]:
    if col in n.carriers.columns:
        n.carriers[col] = n.carriers[col].astype(str).fillna("")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
out_fn.parent.mkdir(parents=True, exist_ok=True)
n.export_to_netcdf(out_fn)

print("\n✓ EI03 Energy Island network saved:")
print(out_fn)


INFO:pypsa.io:Imported network base_s_50_elec.nc has buses, carriers, generators, lines, links, loads, storage_units, stores


Loaded base network: base_s_50_elec.nc
✓ Dispatch-only mode enforced
✓ Energy Island bus added
✓ EI03 AC tie-line added (critical)
✓ Required carriers ensured
✓ Offshore wind added
✓ HVDC links added
✓ Hydrogen bus added
✓ Electrolyser added
✓ H2 storage added
✓ H2 export load added (THIS ENABLES ELECTROLYSIS)
✓ Battery added
✓ Load shedding ensured


INFO:pypsa.io:Exported network 'EI03_energy_island.nc' contains: storage_units, generators, buses, stores, lines, carriers, loads, links



✓ EI03 Energy Island network saved:
C:\Users\franc\pypsa\pypsa-eur\results\networks\EI03_energy_island.nc


In [3]:
############################################## DEBUGGER ##############################################
import pypsa
import pandas as pd
from pathlib import Path

n = pypsa.Network(
    Path(r"C:\Users\franc\pypsa\pypsa-eur\results\networks\EI03_energy_island.nc")
)

n.snapshots = n.snapshots[:72]
n.optimize()

print("Wind p_max_pu summary:")
print(p_max_pu_ns.describe())

WIND_THRESHOLD = p_max_pu_ns.quantile(0.80)
print("WIND_THRESHOLD:", WIND_THRESHOLD)

print("H2 active share:",
      h2_active.mean())

print("H2 active hours:",
      h2_active.sum())

INFO:pypsa.io:Imported network EI03_energy_island.nc has buses, carriers, generators, lines, links, loads, storage_units, stores
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|███████████████████████████████████████████████████| 10/10 [00:00<00:00, 39.31it/s]
INFO:linopy.io: Writing time: 1.51s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 101260 primals, 209908 duals
Objective: 1.51e+09
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Link-fix-p-lower, Link-fix-p-upper, Store-fix-e-lower, Store-fix-e-upper, Store-ext-e-lower, Store-ext-e-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lowe

Wind p_max_pu summary:
count    8760.000000
mean        0.504022
std         0.225923
min         0.030690
25%         0.305894
50%         0.516393
75%         0.704439
max         0.881296
dtype: float64
WIND_THRESHOLD: 0.7436392018346442
H2 active share: 0.2
H2 active hours: 1752
